**Step : 1. Loading the Infrastructure (The Environment Setup)**

This imports the foundational toolkits. We pull PyTorch for the heavy machine learning tensor math, Hugging Face datasets to fetch our raw movie reviews instantly, and Hugging Face transformers to load our industrial AI architectures.

In [1]:
# =====================================================================
# STEP 1: INSTALL REQUIRED ECOSYSTEM LIBRARIES
# =====================================================================
# If running for the first time in a new Colab cell, uncomment and execute:
# !pip install transformers datasets accelerate

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from tqdm.auto import tqdm

**Step 2: The Training Configurations (Hyperparameters)**

• device detects and boots up our Google Colab T4 GPU graphics card.

• batch_size tells the system to read 16 movie reviews at a time before updating weights.

• temperature scales our probability blurring to open up the "Dark Knowledge".

• alpha_soft sets a 70/30 split, forcing the student to prioritize learning from the teacher over the textbook's hard answers.

In [2]:
# =====================================================================
# STEP 2: GLOBAL HYPERPARAMETERS (The Training Arguments)
# =====================================================================
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size   = 16          # Flashcard Stack Size
max_len      = 128         # Trim/Pad texts uniformly
epochs       = 1           # Complete iterations through textbook
lr           = 3e-5        # Study Step Size
temperature  = 2.0         # Blurry filter slider to expand dark knowledge
alpha_soft   = 0.7         # Focus distribution: 70% Teacher, 30% Hard Labels

**Step 3: Feeding the Models (Data Pipeline & Tokenization)**
1. Downloads the IMDb Movie Reviews dataset. We slice a clean subset (2,000 for training, 500 for testing) to ensure the code executes in under 5 minutes during class.
2. Loads a Tokenizer. Computers cannot read words like "masterpiece" or "trash". The tokenizer converts words into matching numerical ID numbers.
3. DataCollatorWithPadding dynamically adds trailing zeroes (padding) to short sentences in a batch so they all form a perfect, uniform mathematical matrix.
4. DataLoader packs the tokenized sequences into ready-to-process batches.

In [5]:
# =====================================================================
# STEP 3: DATA PREPARATION & TOKENIZATION PIPELINE
# =====================================================================
print("Downloading and caching IMDb Movie Reviews dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

# Select a fast-running subset for the classroom lab window
train_subset = raw_datasets["train"].shuffle(seed=42).select(range(2000))
test_subset  = raw_datasets["test"].shuffle(seed=42).select(range(500))

# Initialize common tokenizer matching our Teacher-Student models
teacher_id = "distilbert-base-uncased"
student_id = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer  = AutoTokenizer.from_pretrained(teacher_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_len)

print("Tokenizing and mapping texts into memory layers...")
tokenized_train = train_subset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test  = test_subset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator manages the dynamic zero-padding arrays per batch execution
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

train_loader = DataLoader(tokenized_train, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
test_loader  = DataLoader(tokenized_test, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing and mapping texts into memory layers...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

**Step 4: Initializing the Master and Apprentice**

	1. We define num_labels = 2 because this is binary sentiment analysis (Positive or Negative).
	2. We load DistilBERT (66M parameters) to act as our expert Teacher, and an ultra-compact TinyBERT (14.5M parameters) to act as our Apprentice Student.
	3. The Lock Layer: We call teacher.eval() and run a for loop setting requires_grad = False. This completely freezes the teacher's parameters. We are only printing out the teacher's expert notes, not changing its brain.


In [6]:
# =====================================================================
# STEP 4: INSTANTIATE AND CONFIGURING MODEL INTERACTION
# =====================================================================
num_labels = 2  # Binary classifications: Positive (1) vs Negative (0)
print(f"Initializing Expert Teacher ({teacher_id})...")
teacher = AutoModelForSequenceClassification.from_pretrained(teacher_id, num_labels=num_labels).to(device)

print(f"Initializing Apprentice Student ({student_id})...")
student = AutoModelForSequenceClassification.from_pretrained(student_id, num_labels=num_labels).to(device)

# Mandatory Protection Step: Lock the expert professor completely
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

Initializing Expert Teacher (distilbert-base-uncased)...


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Initializing Apprentice Student (huawei-noah/TinyBERT_General_4L_312D)...


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 62.7MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

**Step 5: Preparing the Two Loss Judges**
1. optimizer handles updating the student model's parameters based on its errors.
2. ce_loss initializes Judge 1 (Cross-Entropy) to measure standard hard label performance.
3. kl_loss initializes Judge 2 (KL-Divergence) to evaluate how well the student's probability curve mimics the teacher's curve shape.

In [7]:
# =====================================================================
# STEP 5: SETUP OPTIMIZER & MATRIC OBJECTIVES (The Dual Judges)
# =====================================================================
optimizer = optim.AdamW(student.parameters(), lr=lr)
ce_loss   = nn.CrossEntropyLoss()
kl_loss   = nn.KLDivLoss(reduction="batchmean", log_target=True)

**Step 6: The Core Distillation Training Loop**

1. We send the tokenized batch inputs to the GPU device memory.
2. with torch.no_grad(): ensures the teacher predicts text features without wasting memory tracking training math. We divide t_logits / temperature to flatten out the predictions and reveal the "Dark Knowledge". We use log_softmax to keep our numbers mathematically stable.
3. The student processes the exact same text and generates its own log soft shapes.
4. We calculate loss_hard (textbook answers alignment) and loss_soft (teacher curve alignment). We multiply loss_soft by temperature ** 2 to restore gradient scale lost during division.
5. We blend the losses using our Alpha ratio slider.
6. loss.backward() calculates the math penalties, and optimizer.step() moves forward to tweak the student model's weights.

In [8]:
# =====================================================================
# STEP 6: CORE KNOWLEDGE DISTILLATION LOOP
# =====================================================================
def distill_epoch(epoch_idx):
    student.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_idx}")

    for batch in pbar:
        # Move tensor arrays to Colab T4 GPU
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 1. Capture Master's reasoning configurations inside log space
        with torch.no_grad():
            t_logits = teacher(input_ids, attention_mask=attention_mask).logits
            t_log_soft = torch.log_softmax(t_logits / temperature, dim=-1)

        # 2. Forward pass calculation on learning student model
        s_logits = student(input_ids, attention_mask=attention_mask).logits
        s_log_soft = torch.log_softmax(s_logits / temperature, dim=-1)

        # 3. Compute both Judge Loss evaluations
        loss_hard = ce_loss(s_logits, labels)
        loss_soft = kl_loss(s_log_soft, t_log_soft) * (temperature ** 2)

        # 4. Blended optimization objective
        loss = (alpha_soft * loss_soft) + ((1.0 - alpha_soft) * loss_hard)

        # 5. Backpropagation step tracking exclusively on student architecture
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    print(f"Epoch {epoch_idx} Finished | Average Combined Loss: {total_loss / len(train_loader):.4f}")

**Step 7 & 8: Evaluation Testing & Disk Storage**
This runs our final test. It disables training weights (model.eval()), evaluates the student against our separate validation dataset split, computes the overall accuracy score, and prints it cleanly to your console logs.

In [9]:
# =====================================================================
# STEP 7: VALIDATION EVALUATION CRITERIA
# =====================================================================
def evaluate_model(model, data_loader, name="Model"):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask=attention_mask).logits
            predictions = torch.argmax(logits, dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = (correct / total) * 100
    print(f"{name} Evaluation Accuracy: {accuracy:.2f}%")
    return accuracy

In [10]:
# =====================================================================
# STEP 8: RUN EXPERIMENT EXECUTIONS
# =====================================================================
print("\n--- Starting Laboratory Diagnostics ---")
for epoch in range(1, epochs + 1):
    distill_epoch(epoch)
    evaluate_model(student, test_loader, "Distilled Student")

# Save final parameters out to disk dump files
student.save_pretrained("distilled_tinybert_imdb")
tokenizer.save_pretrained("distilled_tinybert_imdb")
print("\nStudent parameters saved cleanly inside 'distilled_tinybert_imdb/'.")



--- Starting Laboratory Diagnostics ---


Epoch 1:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1 Finished | Average Combined Loss: 0.1950
Distilled Student Evaluation Accuracy: 76.20%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Student parameters saved cleanly inside 'distilled_tinybert_imdb/'.


In [11]:
# =====================================================================
# STEP 9: LIVE INFERENCE TESTING ON TRICKY EDGE-CASES
# =====================================================================
def predict_live_sentiment(text_input, model, tokenizer_instance, temp=1.0):
    model.eval()

    # 1. Tokenize the live user input string
    inputs = tokenizer_instance(text_input, return_tensors="pt", truncation=True, max_length=128).to(device)

    # 2. Forward pass without tracking training gradients
    with torch.no_grad():
        logits = model(**inputs).logits
        # Apply temperature to see the underlying soft probability distribution
        probabilities = torch.softmax(logits / temp, dim=-1).squeeze().cpu().tolist()

    prediction_idx = torch.argmax(logits, dim=-1).item()
    sentiment_label = "POSITIVE " if prediction_idx == 1 else "NEGATIVE "

    print(f"\n Review: \"{text_input}\"")
    print(f" Prediction: {sentiment_label}")
    print(f" Confidence Vector -> [ Negative: {probabilities[0]:.4f}, Positive: {probabilities[1]:.4f} ]")

# --- Test Suite of Complex Reviews ---
test_reviews = [
    "I guess this masterpiece wasn't completely terrible.",           # Highly sarcastic/double negation
    "The acting was brilliant, but the plot line was absolute trash.", # Mixed sentiment
    "I fell asleep halfway through, but the visual effects were okay.", # Weak sentiment
    "Do not miss this if you want to waste two hours of your life."    # Idiomatic trap word ("Do not miss")
]

print("\n--- Running Live Inference Test Suite on Distilled Student ---")
for review in test_reviews:
    # We pass temp=2.0 so students can explicitly see the fluid "soft" logic curves
    predict_live_sentiment(review, student, tokenizer, temp=2.0)



--- Running Live Inference Test Suite on Distilled Student ---

 Review: "I guess this masterpiece wasn't completely terrible."
 Prediction: POSITIVE 
 Confidence Vector -> [ Negative: 0.4704, Positive: 0.5296 ]

 Review: "The acting was brilliant, but the plot line was absolute trash."
 Prediction: POSITIVE 
 Confidence Vector -> [ Negative: 0.4899, Positive: 0.5101 ]

 Review: "I fell asleep halfway through, but the visual effects were okay."
 Prediction: POSITIVE 
 Confidence Vector -> [ Negative: 0.4748, Positive: 0.5252 ]

 Review: "Do not miss this if you want to waste two hours of your life."
 Prediction: POSITIVE 
 Confidence Vector -> [ Negative: 0.4823, Positive: 0.5177 ]
